In [1]:
import itertools
import numpy as np


def get_input_data():
    players_input = input("Enter students (comma-separated): ").strip()
    players = [p.strip() for p in players_input.split(",") if p.strip()]

    rooms_input = input("Enter rooms (comma-separated): ").strip()
    rooms = [r.strip() for r in rooms_input.split(",") if r.strip()]

    if len(players) == 0 or len(rooms) == 0:
        raise ValueError("Player and room lists cannot be empty.")

    if len(rooms) != len(players):
        raise ValueError("The number of rooms must match the number of students.")

    rents = {}
    for room in rooms:
        while True:
            try:
                val_in = input(f"Enter rent for {room}: ").strip()
                if not val_in:
                    continue
                rents[room] = float(val_in)
                break
            except ValueError:
                pass

    valuations = {player: {} for player in players}
    for player in players:
        for room in rooms:
            while True:
                try:
                    val_in = input(f"Enter valuation of {room} for {player}: ").strip()
                    if not val_in:
                        continue
                    valuations[player][room] = float(val_in)
                    break
                except ValueError:
                    pass

    return players, rooms, valuations, rents


def find_nash_numpy(players, rooms, valuations, rents):
    n = len(players)

    tensor_shape = tuple([n] * n + [n])
    payoff_matrix = np.zeros(tensor_shape)

    for profile in itertools.product(range(n), repeat=n):
        for player_idx in range(n):
            chosen_room_idx = profile[player_idx]
            chosen_room_name = rooms[chosen_room_idx]

            val = valuations[players[player_idx]][chosen_room_name]
            rent = rents[chosen_room_name]

            payoff_matrix[profile + (player_idx,)] = val - rent

    is_nash = np.ones(tuple([n] * n), dtype=bool)

    for player_idx in range(n):
        view_shape = list(range(n))
        view_shape.remove(player_idx)
        view_shape.insert(0, player_idx)

        transposed_payoffs = np.transpose(
            payoff_matrix[..., player_idx], view_shape
        )

        max_payoffs = np.max(transposed_payoffs, axis=0)
        player_best_response = transposed_payoffs == max_payoffs

        inv_view_shape = np.argsort(view_shape)
        is_nash &= np.transpose(player_best_response, inv_view_shape)

    nash_indices = np.argwhere(is_nash)

    print("\n=======================================================")
    print("               NASH EQUILIBRIUM ANALYSIS               ")

    found_stable_allocation = False

    for idx_profile in list(nash_indices):
        if len(set(idx_profile)) == n:
            found_stable_allocation = True
            print("Optimal Strategy Profile:")
            print("-" * 40)
            payoffs = []
            for p_idx, r_idx in enumerate(idx_profile):
                p_name = players[p_idx]
                r_name = rooms[r_idx]
                utility = payoff_matrix[tuple(idx_profile) + (p_idx,)]
                payoffs.append(int(utility))
                print(f"  {p_name:10} ->  {r_name} (Utility: {int(utility)})")
            print("-" * 40)
            print(f"Payoff Vector: {tuple(payoffs)}")
            print("Status: Stable allocation identified.")

    if not found_stable_allocation:
        print("No pure strategy Nash equilibrium with unique allocation found.")
    print("=======================================================")


if __name__ == "__main__":
    try:
        players, rooms, valuations, rents = get_input_data()
        find_nash_numpy(players, rooms, valuations, rents)
    except KeyboardInterrupt:
        pass
    except Exception as e:
        print(f"Error: {e}")

Enter students (comma-separated):  Malika
Enter rooms (comma-separated):  2
Enter rent for 2:  200
Enter valuation of 2 for Malika:  200



               NASH EQUILIBRIUM ANALYSIS               
Optimal Strategy Profile:
----------------------------------------
  Malika     ->  2 (Utility: 0)
----------------------------------------
Payoff Vector: (0,)
Status: Stable allocation identified.
